# 📖 Notebook 1: Data Residency Basics

**Goal**: Understand why data must stay in specific geographic regions and how to route writes to the correct database.

## Learning Objectives

By the end of this notebook, you'll understand:
- What PII (Personally Identifiable Information) is and why it matters
- What data residency means under GDPR
- How Azure Paired Regions keep data within legal boundaries
- How to implement geo-routing — sending user data to the right regional database

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/gdpr-paired-regions
docker-compose up -d
```

### Visualization

- **Adminer** (Database GUI): http://localhost:8081  
  - EU-West: Server `postgres-eu-west`, User `demo`, Password `demo`, DB `gdpr_eu_west`
  - EU-North: Server `postgres-eu-north`, User `demo`, Password `demo`, DB `gdpr_eu_north`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
from datetime import datetime

# ── Connection settings for both regions ────────────────────
# In a real Azure deployment, these would be different Azure
# Database for PostgreSQL instances in different regions.
# Here we simulate with two Docker containers on different ports.

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 5434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

def get_connection(region):
    """Get a database connection for the specified region."""
    config = EU_WEST_CONFIG if region == "eu-west" else EU_NORTH_CONFIG
    return psycopg2.connect(**config)

# Test both connections
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    count = cur.fetchone()[0]
    print(f"✅ Connected to {region} — {count} users found")
    conn.close()

✅ Connected to eu-west — 15 users found
✅ Connected to eu-north — 15 users found


## 1. What is PII?

**PII** stands for **Personally Identifiable Information** — any data that can identify a specific person.

Under GDPR, this includes:

| Data | Why It's PII |
|------|--------------|
| Full name | Directly identifies someone |
| Email address | Directly identifies someone |
| Phone number | Can be linked to a person |
| Date of birth | Combined with name, uniquely identifies |
| IP address | Can be traced to a household |
| Street address | Physical location of a person |
| Order history | Reveals personal preferences and habits |

Let's look at the PII we have in our database:

In [2]:
# Show all PII columns for users in EU-West
conn = get_connection("eu-west")
cur = conn.cursor()

cur.execute("""
    SELECT id, email, full_name, phone, date_of_birth, country_code, home_region
    FROM users
    WHERE home_region = 'eu-west'
    ORDER BY id
""")

print("🇳🇱 EU-West Region Users (PII Data)")
print("=" * 100)
print(f"{'ID':<4} {'Email':<32} {'Name':<20} {'Phone':<20} {'DOB':<12} {'Country'}")
print("-" * 100)
for row in cur.fetchall():
    print(f"{row[0]:<4} {row[1]:<32} {row[2]:<20} {row[3] or 'N/A':<20} {str(row[4]):<12} {row[5]}")

print("\n⚠️  Every column above except 'ID' is PII under GDPR!")
print("    This data MUST stay within the EU.")
conn.close()

🇳🇱 EU-West Region Users (PII Data)
ID   Email                            Name                 Phone                DOB          Country
----------------------------------------------------------------------------------------------------
1    anna.devries@example.nl          Anna de Vries        +31-6-1234-5678      1990-03-15   NL
2    marc.dupont@example.be           Marc Dupont          +32-2-555-0123       1985-07-22   BE
3    claire.martin@example.fr         Claire Martin        +33-1-4567-8901      1992-11-08   FR
4    hans.mueller@example.de          Hans Müller          +49-30-9876-5432     1978-05-30   DE
5    sophie.jansen@example.nl         Sophie Jansen        +31-6-8765-4321      1995-09-12   NL
6    pierre.leblanc@example.fr        Pierre Leblanc       +33-6-1111-2222      1988-12-25   FR
7    lena.schmidt@example.de          Lena Schmidt         +49-89-3333-4444     1993-02-18   DE
8    jan.bakker@example.nl            Jan Bakker           +31-6-5555-6666      1982-08-04 

## 2. What is Data Residency?

**Data residency** means the physical location where data is stored. Under GDPR:

- EU citizen data should stay in the **EU/EEA** (European Economic Area)
- Transferring data outside the EU requires **special legal mechanisms** (e.g., Standard Contractual Clauses)
- Some countries (like Germany) have even stricter rules requiring data to stay in-country

### Why This Matters for Cloud Computing

When you use a cloud provider like Azure, your data is stored in a **specific data center** in a **specific country**. You need to know:

1. **Where is my primary data?** (e.g., Netherlands)
2. **Where are my backups?** (e.g., Ireland — still in EU ✅)
3. **Where does my data go during failover?** (e.g., Ireland — still in EU ✅)
4. **Can support engineers in other countries see my data?** (needs legal basis)

### Azure Paired Regions Solve This

Microsoft pairs regions **within the same geography**:

```
West Europe (Netherlands) ←→ North Europe (Ireland)   [Geography: Europe]
France Central (Paris)    ←→ France South (Marseille)  [Geography: France]
Germany West (Frankfurt)  ←→ Germany North (Berlin)    [Geography: Germany]
```

**Guarantee**: Data never leaves the geography boundary, even during disaster recovery.

## 3. Geo-Routing: Sending Data to the Right Region

In a real system, when a user signs up or updates their profile, we need to route their data to the correct regional database. This is called **geo-routing**.

### How It Works

```
New User Signs Up
       │
       ▼
  ┌─────────────┐
  │ Geo-Router  │ ← Looks at user's country_code
  └──────┬──────┘
         │
    ┌────┴────┐
    ▼         ▼
 EU-West   EU-North
 NL,BE,    IE,SE,
 FR,DE     FI,DK
```

Let's build this:

In [3]:
# ── Geo-Routing Configuration ──────────────────────────────
# Maps country codes to their assigned Azure paired region.
# In production, this would come from a configuration service
# or Azure Traffic Manager.

COUNTRY_TO_REGION = {
    # EU-West countries (primary: Netherlands)
    "NL": "eu-west",   # Netherlands
    "BE": "eu-west",   # Belgium
    "FR": "eu-west",   # France
    "DE": "eu-west",   # Germany
    "LU": "eu-west",   # Luxembourg
    "AT": "eu-west",   # Austria
    "CH": "eu-west",   # Switzerland

    # EU-North countries (primary: Ireland)
    "IE": "eu-north",  # Ireland
    "SE": "eu-north",  # Sweden
    "FI": "eu-north",  # Finland
    "DK": "eu-north",  # Denmark
    "NO": "eu-north",  # Norway
    "IS": "eu-north",  # Iceland
}

def get_region_for_country(country_code: str) -> str:
    """
    Determines which Azure region should store data for a given country.
    This is the core of geo-routing.
    
    In Azure, this is handled by Azure Traffic Manager or Azure Front Door,
    which routes requests to the nearest compliant region.
    """
    region = COUNTRY_TO_REGION.get(country_code.upper())
    if region is None:
        raise ValueError(
            f"Country '{country_code}' not mapped to any region. "
            f"This user may need special handling (non-EU data residency)."
        )
    return region

# Test the geo-router
test_countries = ["NL", "IE", "DE", "SE", "FR", "FI"]
print("🌍 Geo-Routing Table")
print("=" * 40)
for cc in test_countries:
    region = get_region_for_country(cc)
    print(f"  {cc} → {region}")

🌍 Geo-Routing Table
  NL → eu-west
  IE → eu-north
  DE → eu-west
  SE → eu-north
  FR → eu-west
  FI → eu-north


## 🚫 Bad → ✅ Best: Why Geo-Routing Matters

Before we celebrate the geo-router, let's see what happens **without** it.
Many small teams start by putting *all* users in one database — usually in the
cheapest region (often US-East). Let's see why that breaks GDPR.

### ❌ Bad: One global database in a non-EU region

```python
# Everyone goes into the same US database
def create_user_BAD(email, country_code):
    conn = connect(us_east_db)             # US region
    conn.execute("INSERT INTO users ...")  # EU citizen's PII now in US 🇺🇸
```

What's wrong?
1. **Schrems II ruling (2020)** — the EU Court of Justice invalidated the
   "Privacy Shield" framework. Transfers of EU personal data to the US without
   extra safeguards are **illegal** by default.
2. **No data residency guarantee** — German banking/health rules require data
   to stay *in Germany*, not just "somewhere".
3. **One fire = total loss** — no paired-region DR.

### ⚠️ Slightly-less-bad: One EU database, no paired region

Better, but if the single EU datacenter goes down, you're offline and your
backups (if any) might be in a non-compliant region.

### ✅ Best: Geo-routed writes + paired region DR

The `create_user_in_correct_region()` function we're about to build is the
"best" version:
- Routes writes by `country_code` to the **correct EU region**
- Logs residency for the audit trail
- The paired region (next notebook) provides DR **without** leaving the EU


In [4]:
# ── Demo: the BAD way — ignoring the user's country ─────────
# This is the anti-pattern we want to AVOID.

def create_user_BAD(email, full_name, country_code):
    """
    Anti-pattern: always writes to eu-west, ignoring the user's country.
    This is how many teams start — one database, one region.
    It breaks down as soon as you have:
      • a German customer who legally needs data in Germany
      • a regulator asking "where exactly is Ms Svensson's phone number?"
      • a regional outage (no DR)
    """
    conn = get_connection("eu-west")   # ← hard-coded, no routing
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO users (email, full_name, country_code, home_region, consent_given)
        VALUES (%s, %s, %s, 'eu-west', TRUE)
        ON CONFLICT (email) DO NOTHING
        RETURNING id
    """, (email, full_name, country_code))
    row = cur.fetchone()
    conn.commit(); conn.close()
    return row[0] if row else None

# Pretend a Swedish user signs up — naively we put them in eu-west
bad_id = create_user_BAD("bad.ingrid@example.se", "Bad Ingrid", "SE")
print(f"❌ BAD: Swedish user stored in eu-west (id={bad_id})")
print("   → An auditor would ask: 'Her country is SE but home_region is eu-west — why?'\n")

# ── Now compare with the ✅ BEST way (built in the next cells) ──
print("✅ BEST: use get_region_for_country() + create_user_in_correct_region()")
print(f"   A Swedish user would be routed to: {get_region_for_country('SE')}")

# Clean up the bad insert so it does not pollute the rest of the notebook
conn = get_connection("eu-west")
conn.cursor().execute("DELETE FROM users WHERE email = 'bad.ingrid@example.se'")
conn.commit(); conn.close()
print("\n🧹 Cleaned up the bad-practice user.")


❌ BAD: Swedish user stored in eu-west (id=16)
   → An auditor would ask: 'Her country is SE but home_region is eu-west — why?'

✅ BEST: use get_region_for_country() + create_user_in_correct_region()
   A Swedish user would be routed to: eu-north

🧹 Cleaned up the bad-practice user.


In [5]:
def create_user_in_correct_region(email, full_name, phone, date_of_birth, country_code):
    """
    Creates a new user in the geographically correct database.
    
    This is how a GDPR-compliant system works:
    1. Determine the user's region from their country
    2. Connect to that region's database
    3. Insert the user data
    4. Log the data residency action for audit
    """
    # Step 1: Determine the correct region
    region = get_region_for_country(country_code)
    print(f"📍 User from {country_code} → routing to {region}")

    # Step 2: Connect to the correct regional database
    conn = get_connection(region)
    cur = conn.cursor()

    try:
        # Step 3: Insert the user
        cur.execute("""
            INSERT INTO users (email, full_name, phone, date_of_birth, country_code, home_region, consent_given, consent_date)
            VALUES (%s, %s, %s, %s, %s, %s, TRUE, NOW())
            RETURNING id
        """, (email, full_name, phone, date_of_birth, country_code, region))
        user_id = cur.fetchone()[0]

        # Step 4: Log the data residency action
        cur.execute("""
            INSERT INTO data_residency_log (user_id, action, source_region, table_name, record_id, reason)
            VALUES (%s, 'write', %s, 'users', %s, 'New user registration — geo-routed by country code')
        """, (user_id, region, user_id))

        # Also log consent
        cur.execute("""
            INSERT INTO consent_log (user_id, action, purpose, ip_address)
            VALUES (%s, 'granted', 'essential', '127.0.0.1')
        """, (user_id,))

        conn.commit()
        print(f"✅ User '{full_name}' created with ID {user_id} in {region}")
        print(f"   📋 Data residency logged | Consent recorded")
        return user_id, region

    except Exception as e:
        conn.rollback()
        print(f"❌ Error: {e}")
        raise
    finally:
        conn.close()


# ── Demo: Create users from different countries ──────────────
print("\n🆕 Creating New Users with Geo-Routing")
print("=" * 50)

# A German user → should go to EU-West (Netherlands data center)
create_user_in_correct_region(
    email="max.weber@example.de",
    full_name="Max Weber",
    phone="+49-30-5555-1234",
    date_of_birth="1991-04-20",
    country_code="DE"
)

print()

# A Swedish user → should go to EU-North (Ireland data center)
create_user_in_correct_region(
    email="ingrid.svensson@example.se",
    full_name="Ingrid Svensson",
    phone="+46-8-555-9876",
    date_of_birth="1994-08-15",
    country_code="SE"
)


🆕 Creating New Users with Geo-Routing
📍 User from DE → routing to eu-west
✅ User 'Max Weber' created with ID 17 in eu-west
   📋 Data residency logged | Consent recorded

📍 User from SE → routing to eu-north
✅ User 'Ingrid Svensson' created with ID 16 in eu-north
   📋 Data residency logged | Consent recorded


(16, 'eu-north')

In [6]:
# ── Verify: Check that data landed in the correct region ────

print("🔍 Verifying Data Residency")
print("=" * 60)

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()

    # Check for our newly created users
    cur.execute("""
        SELECT email, full_name, country_code, home_region
        FROM users
        WHERE email IN ('max.weber@example.de', 'ingrid.svensson@example.se')
    """)
    results = cur.fetchall()

    print(f"\n📦 {region.upper()} database:")
    if results:
        for row in results:
            print(f"   ✅ Found: {row[1]} ({row[0]}) — country={row[2]}, region={row[3]}")
    else:
        print(f"   (no matching users — correct! They belong to the other region)")

    conn.close()

print("\n💡 Notice: Each user's data exists ONLY in their assigned region.")
print("   Max Weber (DE) → EU-West only")
print("   Ingrid Svensson (SE) → EU-North only")
print("   This is data residency in action!")

🔍 Verifying Data Residency



📦 EU-WEST database:
   ✅ Found: Max Weber (max.weber@example.de) — country=DE, region=eu-west

📦 EU-NORTH database:
   ✅ Found: Ingrid Svensson (ingrid.svensson@example.se) — country=SE, region=eu-north

💡 Notice: Each user's data exists ONLY in their assigned region.
   Max Weber (DE) → EU-West only
   Ingrid Svensson (SE) → EU-North only
   This is data residency in action!


## 4. Why Does Microsoft Use This Pattern?

Microsoft Azure is one of the biggest cloud providers in the world. They serve EU governments, banks, hospitals, and enterprises that **legally cannot** store data outside the EU.

### The Business Case

- **EU public sector** contracts require data sovereignty (€ billions in deals)
- **German data protection** is among the strictest in the world
- **Financial regulations** (like PSD2) require data to stay in-jurisdiction
- **Healthcare data** (HIPAA-equivalent in EU) must not leave country borders

### How Azure Implements It

1. **Azure Traffic Manager** routes requests to the nearest compliant region
2. **Azure SQL Geo-Replication** copies data between paired regions only
3. **Azure Policy** enforces that resources can only be created in approved regions
4. **Azure Compliance Manager** provides built-in GDPR compliance scoring

### What We Simulated

In this notebook, our `get_region_for_country()` function simulates what Azure Traffic Manager does — routing data to the correct regional database based on the user's geography.

In [7]:
# ── Check data residency logs ──────────────────────────────

print("📋 Data Residency Audit Log")
print("=" * 80)

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        SELECT drl.user_id, u.full_name, drl.action, drl.source_region,
               drl.table_name, drl.reason, drl.created_at
        FROM data_residency_log drl
        LEFT JOIN users u ON drl.user_id = u.id
        ORDER BY drl.created_at DESC
        LIMIT 5
    """)

    print(f"\n📦 {region.upper()} — Recent Residency Events:")
    for row in cur.fetchall():
        print(f"   [{row[6]}] User {row[1] or row[0]}: {row[2]} in {row[3]} ({row[4]})")
        print(f"     Reason: {row[5]}")
    conn.close()

print("\n💡 These logs are essential for GDPR compliance audits.")
print("   They prove that data was stored in the correct region.")

📋 Data Residency Audit Log

📦 EU-WEST — Recent Residency Events:
   [2026-04-20 19:31:36.850263] User Max Weber: write in eu-west (users)
     Reason: New user registration — geo-routed by country code

📦 EU-NORTH — Recent Residency Events:
   [2026-04-20 19:31:36.864959] User Ingrid Svensson: write in eu-north (users)
     Reason: New user registration — geo-routed by country code

💡 These logs are essential for GDPR compliance audits.
   They prove that data was stored in the correct region.


## 5. What Happens with Non-EU Users?

GDPR applies to EU citizens, but what about users from the US, Asia, etc.?

In a real system, you'd have additional region mappings:
- US users → US East + US West paired regions
- Asian users → East Asia + Southeast Asia paired regions
- etc.

The key insight: **the architecture is the same everywhere** — pair regions within the same legal jurisdiction.

In [8]:
# ── Demo: What happens with an unmapped country? ───────────

try:
    region = get_region_for_country("US")
except ValueError as e:
    print(f"⚠️  {e}")
    print()
    print("In a real system, you would:")
    print("1. Route US users to a US-based paired region (e.g., US East + US Central)")
    print("2. Apply US-specific data protection rules (CCPA, etc.)")
    print("3. Never mix US and EU data without explicit legal basis")

⚠️  Country 'US' not mapped to any region. This user may need special handling (non-EU data residency).

In a real system, you would:
1. Route US users to a US-based paired region (e.g., US East + US Central)
2. Apply US-specific data protection rules (CCPA, etc.)
3. Never mix US and EU data without explicit legal basis


## 🎯 Key Takeaways

1. **PII is any data that can identify a person** — names, emails, phones, addresses, even IP addresses
2. **Data residency = physical location of data** — GDPR requires EU data to stay in the EU
3. **Geo-routing** sends writes to the correct regional database based on user location
4. **Azure Paired Regions** guarantee data stays within the same geography, even during DR
5. **Audit logs** prove compliance — always log where data was written and why

## ⏭️ Next Up

In **Notebook 2**, we'll explore how data gets replicated between paired regions for disaster recovery — without leaving the EU.